# KrishiMitra — Market Price Forecaster (Model 3)

OCI Data Science notebook for the **Market Price Predictor** (Module 3).

- **Algorithm:** Prophet (with SARIMA as a documented fallback)
- **Input:** 365-day price history per crop-mandi, seasonality, harvest calendar
- **Output:** next 30-day price forecast per crop-mandi pair -> `ML_PREDICTIONS`
- **Retraining:** OCI Data Science Pipeline (weekly)
- **Serving:** batch prediction job; results written to Oracle ATP `ML_PREDICTIONS`

> Pulls history from `MANDI_PRICES` in Oracle ATP. A synthetic series is used
> here so the notebook runs without DB access; swap `load_price_history()`.

In [ ]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
HORIZON_DAYS = 30

## 1. Load price history
Production query against Oracle ATP:
```python
import oracledb
with oracledb.connect(user=..., password=..., dsn=...) as conn:
    hist = pd.read_sql(
        """SELECT recorded_date AS ds, price_per_qtl AS y
             FROM mandi_prices
            WHERE crop_id = :c AND mandi_name = :m
            ORDER BY recorded_date""", conn, params={'c': crop_id, 'm': mandi})
```

In [ ]:
def load_price_history(days=400, base=2200.0):
    dates = pd.date_range(end=pd.Timestamp.today().normalize(), periods=days, freq='D')
    t = np.arange(days)
    seasonal = 200 * np.sin(2 * np.pi * t / 365.0)        # annual cycle
    weekly = 30 * np.sin(2 * np.pi * t / 7.0)             # mild weekly cycle
    trend = 0.5 * t                                       # gentle upward trend
    noise = np.random.normal(0, 40, days)
    y = (base + trend + seasonal + weekly + noise).clip(300)
    return pd.DataFrame({'ds': dates, 'y': y})

hist = load_price_history()
hist.tail()

## 2. EDA

In [ ]:
import matplotlib.pyplot as plt
print(hist['y'].describe())
plt.figure(figsize=(10, 3))
plt.plot(hist['ds'], hist['y'])
plt.title('Historical price (per qtl)')
plt.tight_layout(); plt.show()

## 3. Fit Prophet + 30-day forecast

In [ ]:
from prophet import Prophet

m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False,
            changepoint_prior_scale=0.05, interval_width=0.8)
# Harvest-calendar regressor could be added via m.add_regressor(...)
m.fit(hist)
future = m.make_future_dataframe(periods=HORIZON_DAYS)
fcst = m.predict(future)
fcst[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(HORIZON_DAYS).head()

## 4. Backtest (walk-forward) — MAPE / RMSE

In [ ]:
from sklearn.metrics import mean_squared_error

split = len(hist) - HORIZON_DAYS
train, test = hist.iloc[:split], hist.iloc[split:]
mb = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
mb.fit(train)
pred = mb.predict(test[['ds']])['yhat'].values
rmse = mean_squared_error(test['y'].values, pred, squared=False)
mape = np.mean(np.abs((test['y'].values - pred) / test['y'].values)) * 100
print(f'Backtest RMSE: {rmse:.2f} | MAPE: {mape:.2f}%')

## 5. Persist forecast to ML_PREDICTIONS (batch job)

In [ ]:
horizon = fcst[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(HORIZON_DAYS).copy()
horizon['model_type'] = 'PRICE'
horizon['unit'] = 'INR/qtl'
horizon['model_version'] = 'prophet-v1'
# Confidence proxy from the prediction interval width.
rng = (horizon['yhat_upper'] - horizon['yhat_lower']).replace(0, np.nan)
horizon['confidence_pct'] = (100 * (1 - (rng / horizon['yhat']).clip(0, 1))).round(2)
horizon.rename(columns={'ds': 'prediction_date', 'yhat': 'predicted_value'}, inplace=True)
horizon[['prediction_date', 'predicted_value', 'unit', 'confidence_pct', 'model_version']].head()

# Production write:
# rows = [(fc_id, 'PRICE', float(r.predicted_value), 'INR/qtl', float(r.confidence_pct),
#          r.prediction_date.date(), 'prophet-v1') for r in horizon.itertuples()]
# cur.executemany("""INSERT INTO ml_predictions(farmer_crop_id, model_type, predicted_value,
#   unit, confidence_pct, prediction_date, model_version) VALUES (:1,:2,:3,:4,:5,:6,:7)""", rows)
# conn.commit()